# Decentralized Ball-DP end-to-end tutorial

This notebook is a compact synthetic-data tutorial for the decentralized Ball-DP API.
It demonstrates the same canonical finite-prior attack setup used by the convex and nonconvex notebooks, but with a different observation model.

The main point is:

$$
\text{finite-prior setup is shared, but the decentralized observation law is different.}
$$

We reuse the centralized finite-prior helpers to construct a public same-label support

$$
S = \{z_1, \ldots, z_m\} \subseteq B(u,r), \qquad Z \sim \mathrm{Unif}(S),
$$

then attack a decentralized Gaussian observer view using the exact finite-prior Bayes rule.


## 1. Imports and plotting defaults

The notebook uses only synthetic data. For real experiments, replace the synthetic data cell with your actual embeddings and labels; the finite-prior setup and decentralized accounting/attack cells remain the same.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp.accountants.rdp import RdpCurve, rdp_to_dp
from quantbayes.ball_dp.attacks.finite_prior_setup import (
    CandidateSource,
    enrich_attack_result_with_trial,
    find_feasible_replacement_banks,
    make_replacement_trial,
    select_support_from_bank,
    target_positions_for_support,
    trial_true_record,
)
from quantbayes.ball_dp.decentralized.accounting import (
    LocalNodeSGDSchedule,
    account_linear_gaussian_observer,
    account_public_transcript_node_local_rdp,
)
from quantbayes.ball_dp.decentralized.attacks import (
    make_linear_gaussian_mean_fn,
    run_linear_gaussian_finite_prior_attack,
)
from quantbayes.ball_dp.decentralized.gossip import (
    constant_mixing_matrices,
    gossip_observer_noise_covariance,
    gossip_transfer_matrix,
    graph_distances,
    make_graph_adjacency,
    metropolis_mixing_matrix,
    selector_matrix,
)
from quantbayes.ball_dp.decentralized.prototypes import (
    partition_indices_iid,
    run_noisy_prototype_gossip,
)
from quantbayes.ball_dp.evaluation.rero import gaussian_direct_ball_rero_bound
from quantbayes.ball_dp.types import Record

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "legend.frameon": False,
    "font.size": 10.5,
})

BALL_COLOR = "#0072B2"
STD_COLOR = "#D55E00"
BASELINE_COLOR = "#4D4D4D"
ATTACK_COLOR = "#7A5195"
UTILITY_COLOR = "#009E73"


## 2. Synthetic embeddings

We generate clustered embeddings and rescale them to satisfy a public norm bound

$$
\|x\|_2 \le B.
$$

In a real experiment, this is the only cell you need to replace: provide `X_train`, `y_train`, `X_test`, and `y_test` from your embedding pipeline.


In [ ]:
def make_synthetic_embeddings(
    *,
    seed: int = 0,
    n_samples: int = 900,
    n_features: int = 12,
    num_classes: int = 3,
    public_fraction: float = 0.35,
    B: float = 5.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float, int]:
    X, y = make_classification(
        n_samples=int(n_samples),
        n_features=int(n_features),
        n_informative=int(n_features),
        n_redundant=0,
        n_repeated=0,
        n_classes=int(num_classes),
        n_clusters_per_class=1,
        class_sep=2.8,
        flip_y=0.0,
        random_state=int(seed),
    )
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    X_train, X_public, y_train, y_public = train_test_split(
        X,
        y,
        test_size=float(public_fraction),
        random_state=int(seed),
        stratify=y,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_public = scaler.transform(X_public).astype(np.float32)

    max_norm = max(
        float(np.linalg.norm(X_train, axis=1).max()),
        float(np.linalg.norm(X_public, axis=1).max()),
        1e-12,
    )
    scale = 0.95 * float(B) / max_norm
    return (
        (scale * X_train).astype(np.float32),
        y_train.astype(np.int32),
        (scale * X_public).astype(np.float32),
        y_public.astype(np.int32),
        float(B),
        int(num_classes),
    )

SEED = 7
X_train, y_train, X_public, y_public, BOUND_B, NUM_CLASSES = make_synthetic_embeddings(seed=SEED)
FEATURE_DIM = int(X_train.shape[1])

print("Private train:", X_train.shape)
print("Public/eval:", X_public.shape)
print("Feature dim:", FEATURE_DIM)
print("Classes:", sorted(np.unique(y_train).tolist()))
print("Public norm bound B:", BOUND_B)


## 3. Choose a feasible Ball radius

The canonical finite-prior setup needs enough public same-label candidates inside a Ball around a private anchor $u$.
For a demo, we choose $r$ from the empirical distance needed to include at least $m$ same-label public candidates.

The attack prior is then

$$
Z \sim \mathrm{Unif}(S), \qquad |S| = m, \qquad \kappa = \max_i \pi_i = \frac{1}{m}.
$$


In [ ]:
def kth_public_same_label_radius(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_public: np.ndarray,
    y_public: np.ndarray,
    *,
    m: int,
    quantile: float = 0.40,
    multiplier: float = 1.10,
) -> float:
    kth = []
    for x, y in zip(X_train, y_train, strict=True):
        idx = np.flatnonzero(y_public == int(y))
        if idx.size < int(m):
            continue
        d = np.linalg.norm(X_public[idx] - x[None, :], axis=1)
        kth.append(float(np.sort(d)[int(m) - 1]))
    if not kth:
        raise RuntimeError("Could not find any feasible same-label public neighborhoods.")
    return float(multiplier * np.quantile(np.asarray(kth), float(quantile)))

PRIOR_SIZE = 8
RADIUS = kth_public_same_label_radius(
    X_train,
    y_train,
    X_public,
    y_public,
    m=PRIOR_SIZE,
    quantile=0.40,
)
print("Chosen finite-prior radius r:", RADIUS)
print("Oblivious exact-ID baseline 1/m:", 1.0 / PRIOR_SIZE)


## 4. Build the canonical finite-prior support

This is the part shared with the convex and nonconvex tutorials:

1. choose a private anchor $u$;
2. remove it to form $D^-$;
3. build a public same-label bank inside $B(u,r)$;
4. select a support $S \subseteq B(u,r)$;
5. sample a hidden target position in $S$;
6. construct $D^- \cup \{Z\}$.

For the decentralized observer attack below, $D^-$ is not used directly by the Gaussian scorer, but it is still useful because it gives the same canonical replacement-trial object and source-ID bookkeeping used everywhere else.


In [ ]:
public_source = CandidateSource("public", X_public, y_public)

banks = find_feasible_replacement_banks(
    X_train=X_train,
    y_train=y_train,
    candidate_sources=[public_source],
    radius=RADIUS,
    min_support_size=PRIOR_SIZE,
    num_banks=1,
    seed=SEED,
    anchor_selection="large_bank",
    strict=True,
)
bank = banks[0]

support = select_support_from_bank(
    bank,
    m=PRIOR_SIZE,
    selection="farthest",
    seed=SEED,
    draw_index=0,
)

target_pos = target_positions_for_support(
    support,
    policy="sample",
    num_targets=1,
    seed=SEED,
)[0]

trial = make_replacement_trial(
    X_train=X_train,
    y_train=y_train,
    support=support,
    target_support_position=target_pos,
)

print("center source id:", support.center_source_id)
print("center label:", support.center_y)
print("support hash:", support.support_hash)
print("support size:", support.m)
print("target support position:", trial.target_support_position)
print("target source id:", trial.target_source_id)
print("target index in D^- union {target}:", trial.target_index)
print("max distance to center:", float(np.max(support.distances_to_center)))
print("oblivious kappa:", support.oblivious_kappa)

assert np.all(support.distances_to_center <= RADIUS + 1e-5)
assert trial.target_index == len(trial.X_full) - 1


## 5. Visualize the support

The finite-prior support is selected in the original embedding space. The PCA plot is only for visualization.


In [ ]:
pca = PCA(n_components=2, random_state=SEED)
all_points = np.concatenate([X_public, support.center_x[None, :], support.X], axis=0)
Z2 = pca.fit_transform(all_points)
Z_public = Z2[: len(X_public)]
Z_center = Z2[len(X_public)]
Z_support = Z2[len(X_public) + 1 :]

fig, ax = plt.subplots(figsize=(7.2, 5.2))
ax.scatter(Z_public[:, 0], Z_public[:, 1], s=12, alpha=0.25, label="public candidates")
ax.scatter(Z_center[0], Z_center[1], s=150, marker="*", label="center u", color=BASELINE_COLOR)
ax.scatter(Z_support[:, 0], Z_support[:, 1], s=70, label="finite support S", color=ATTACK_COLOR)
ax.scatter(
    Z_support[trial.target_support_position, 0],
    Z_support[trial.target_support_position, 1],
    s=160,
    facecolors="none",
    edgecolors="black",
    linewidths=2.0,
    label="hidden target Z",
)
ax.set_title("Canonical public finite-prior support")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best")
plt.show()


## 6. Decentralized graph and gossip transfer matrix

We now switch from setup to decentralized observation. We use a path graph and Metropolis gossip.

For an attacked node $j$ and observer set $A$, the theorem-side linear Gaussian view is

$$
Y_A = (H_{A \leftarrow j} \otimes I_p) s_j(Z) + \zeta_A,
\qquad
\zeta_A \sim \mathcal{N}(0, K_A \otimes I_p).
$$

Here $H_{A \leftarrow j}$ depends only on the graph, mixing matrices, observer set, and attacked node.


In [ ]:
NUM_NODES = 7
ROUNDS = 6
ATTACKED_NODE = 0
LAZINESS = 0.0

ADJ = make_graph_adjacency("path", num_nodes=NUM_NODES)
W = metropolis_mixing_matrix(ADJ, lazy=LAZINESS)
WS = constant_mixing_matrices(W, num_rounds=ROUNDS)
DIST = graph_distances(ADJ)

fig, ax = plt.subplots(figsize=(7.2, 2.2))
xs = np.arange(NUM_NODES)
ys = np.zeros(NUM_NODES)
for i in range(NUM_NODES - 1):
    ax.plot([xs[i], xs[i + 1]], [0, 0], color="0.65", linewidth=2)
ax.scatter(xs, ys, s=260, color="white", edgecolor="black", zorder=3)
ax.scatter([ATTACKED_NODE], [0], s=280, color=BALL_COLOR, edgecolor="black", zorder=4, label="attacked node")
for i in range(NUM_NODES):
    ax.text(xs[i], 0, str(i), ha="center", va="center", fontsize=10, zorder=5)
ax.set_ylim(-0.4, 0.4)
ax.set_axis_off()
ax.set_title("Path graph used in the decentralized demo")
ax.legend(loc="upper right")
plt.show()

pd.DataFrame(W).round(3)


## 7. Public-transcript bridge accounting

The decentralized public-transcript bridge says that arbitrary deterministic gossip and final post-processing do not add privacy cost beyond the attacked node's local noisy releases.

For a local Poisson-SGD-style schedule, the Ball sensitivity per step is

$$
\Delta_t^{\mathrm{Ball}}(r) = \min\{L_z r, 2 C_t\},
$$

while the standard replacement sensitivity is

$$
\Delta_t^{\mathrm{Std}} = 2C_t.
$$

This table is node-local: it does not depend on which external observer later reads the public transcript.


In [ ]:
CLIP_NORM = 1.0
FEATURE_LIPSCHITZ = 1.0
LOCAL_BATCH_SIZE = 64
PROCESS_NOISE_STD = 5.0
DP_DELTA = 1e-6
ORDERS = tuple(float(v) for v in (2, 3, 4, 5, 8, 16, 32, 64, 128))

local_dataset_size = int(math.ceil(len(X_train) / NUM_NODES))
local_schedule = LocalNodeSGDSchedule.from_noise_multipliers(
    dataset_size=local_dataset_size,
    batch_sizes=(LOCAL_BATCH_SIZE,) * ROUNDS,
    clip_norms=(CLIP_NORM,) * ROUNDS,
    noise_multipliers=PROCESS_NOISE_STD / CLIP_NORM,
    radius=RADIUS,
    lz=FEATURE_LIPSCHITZ,
    orders=ORDERS,
    dp_delta=DP_DELTA,
)

bridge = account_public_transcript_node_local_rdp(
    local_schedule,
    attacked_node=ATTACKED_NODE,
)

def first_dp_epsilon(view: Any) -> float:
    certs = getattr(view, "dp_certificates", None)
    if certs:
        return float(certs[0].epsilon)
    cert = getattr(view, "dp_certificate", None)
    if cert is not None:
        return float(cert.epsilon)
    return float("nan")

bridge_df = pd.DataFrame([
    {
        "view": "Ball local public transcript",
        "epsilon": first_dp_epsilon(bridge.ledger.ball),
        "mechanism": bridge.ledger.ball.mechanism,
    },
    {
        "view": "Standard local public transcript",
        "epsilon": first_dp_epsilon(bridge.ledger.standard),
        "mechanism": bridge.ledger.standard.mechanism,
    },
])
display(bridge_df)


## 8. Observer-specific accounting

A specific observer may see a weaker or stronger Gaussian view depending on graph distance and the gossip transfer matrix.
For a finite exact-ID prior with $
\kappa = 1/m$, a direct Gaussian ReRo-style bound uses the whitened transferred sensitivity

$$
 c_{A \leftarrow j}(r)^2
 = \sup_{\|\delta_t\| \le \Delta_t(r)}
 \left\|(K_A^{-1/2} H_{A \leftarrow j} \otimes I_p)\delta\right\|_2^2.
$$

The exact-ID upper bound is then

$$
\Gamma(\kappa) = \Phi\left(\Phi^{-1}(\kappa) + c_{A \leftarrow j}(r)\right).
$$


In [ ]:
def observer_nodes(mode: str, *, attacked_node: int, distances: np.ndarray) -> tuple[int, ...]:
    mode = str(mode)
    if mode == "self":
        return (int(attacked_node),)
    if mode == "nearest":
        d = distances[int(attacked_node)].copy()
        d[int(attacked_node)] = np.inf
        return (int(np.argmin(d)),)
    if mode == "farthest":
        d = distances[int(attacked_node)].copy()
        d[~np.isfinite(d)] = -np.inf
        return (int(np.argmax(d)),)
    if mode == "all":
        return tuple(range(distances.shape[0]))
    raise ValueError("unknown observer mode")


def block_sensitivities(mechanism: str, *, radius: float, rounds: int, decay: float = 1.0) -> tuple[float, ...]:
    if mechanism == "ball":
        base = min(FEATURE_LIPSCHITZ * float(radius), 2.0 * CLIP_NORM)
    elif mechanism == "standard":
        base = 2.0 * CLIP_NORM
    else:
        raise ValueError("mechanism must be 'ball' or 'standard'")
    return tuple(float(base * (decay ** t)) for t in range(int(rounds)))


def direct_bound_from_accountant(acct: Any, *, kappa: float) -> float:
    c = math.sqrt(max(0.0, float(acct.sensitivity_sq)))
    return float(
        gaussian_direct_ball_rero_bound(
            kappa=float(kappa),
            sensitivity=c,
            sigma=1.0,
        )
    )

observer_modes = ["self", "nearest", "farthest", "all"]
account_rows = []
for mode in observer_modes:
    obs = observer_nodes(mode, attacked_node=ATTACKED_NODE, distances=DIST)
    S_A = selector_matrix(obs, num_nodes=NUM_NODES)
    H = gossip_transfer_matrix(WS, observer_selector=S_A, attacked_node=ATTACKED_NODE)
    K = gossip_observer_noise_covariance(
        WS,
        observer_selector=S_A,
        state_noise_stds=PROCESS_NOISE_STD,
        observation_noise_std=0.0,
        jitter=1e-8,
    )
    for mechanism in ["ball", "standard"]:
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=K,
            block_sensitivities=block_sensitivities(mechanism, radius=RADIUS, rounds=ROUNDS),
            parameter_dim=FEATURE_DIM,
            orders=ORDERS,
            radius=RADIUS,
            dp_delta=DP_DELTA,
            attacked_node=ATTACKED_NODE,
            observer=obs,
            covariance_mode="kron_eye",
            method="auto",
        )
        account_rows.append({
            "mechanism": mechanism,
            "observer_mode": mode,
            "observer_nodes": str(obs),
            "sensitivity_c": math.sqrt(max(0.0, float(acct.sensitivity_sq))),
            "direct_exact_id_bound": direct_bound_from_accountant(acct, kappa=support.oblivious_kappa),
            "dp_epsilon": np.nan if acct.dp_certificate is None else float(acct.dp_certificate.epsilon),
            "exact_sensitivity": bool(acct.exact),
            "method": str(acct.method),
        })

account_df = pd.DataFrame(account_rows)
display(account_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
width = 0.35
x = np.arange(len(observer_modes))
for offset, mechanism, color in [(-width / 2, "ball", BALL_COLOR), (width / 2, "standard", STD_COLOR)]:
    sub = account_df[account_df.mechanism == mechanism].set_index("observer_mode").loc[observer_modes]
    ax.bar(x + offset, sub["direct_exact_id_bound"], width=width, label=mechanism, color=color)
ax.axhline(support.oblivious_kappa, color=BASELINE_COLOR, linestyle="--", label="prior baseline 1/m")
ax.set_xticks(x, observer_modes)
ax.set_ylabel("exact-ID upper bound")
ax.set_title("Observer-specific direct Gaussian exact-ID bounds")
ax.set_ylim(0.0, 1.02)
ax.legend()
plt.show()


## 9. Transferred-sensitivity heatmap

The full matrix below shows the directional graph effect. Entry $(a,j)$ is the whitened sensitivity from attacked node $j$ to single observer $a$ under the Ball sensitivity schedule.


In [ ]:
heat = np.zeros((NUM_NODES, NUM_NODES), dtype=float)
for observer in range(NUM_NODES):
    S_A = selector_matrix((observer,), num_nodes=NUM_NODES)
    K = gossip_observer_noise_covariance(
        WS,
        observer_selector=S_A,
        state_noise_stds=PROCESS_NOISE_STD,
        observation_noise_std=0.0,
        jitter=1e-8,
    )
    for attacked in range(NUM_NODES):
        H = gossip_transfer_matrix(WS, observer_selector=S_A, attacked_node=attacked)
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=K,
            block_sensitivities=block_sensitivities("ball", radius=RADIUS, rounds=ROUNDS),
            parameter_dim=FEATURE_DIM,
            orders=ORDERS,
            radius=RADIUS,
            dp_delta=None,
            attacked_node=attacked,
            observer=(observer,),
            covariance_mode="kron_eye",
            method="auto",
        )
        heat[observer, attacked] = math.sqrt(max(0.0, float(acct.sensitivity_sq)))

fig, ax = plt.subplots(figsize=(6.2, 5.0))
im = ax.imshow(heat, aspect="auto")
ax.set_xlabel("attacked node j")
ax.set_ylabel("observer node a")
ax.set_title("Whitened transferred sensitivity c_{a←j}(r)")
fig.colorbar(im, ax=ax, label="sensitivity c")
plt.show()


## 10. Exact finite-prior attack on the decentralized Gaussian view

The decentralized finite-prior attack uses the same support $S$ as the other tutorials, but scores a different observation:

$$
\log p_i
= \log \pi_i
- \frac{1}{2}\left\|K_A^{-1/2}\left(Y_A - \mu_A(z_i)\right)\right\|_2^2.
$$

The MAP estimate is

$$
\hat{i} = \arg\max_i \log p_i.
$$

We enrich the attack result with the canonical source-ID bookkeeping, so exact-ID can be reported by `source_id` rather than only by floating-point feature equality.


In [ ]:
def clip_l2_vector(x: np.ndarray, clip_norm: float) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(x))
    if norm <= float(clip_norm) or norm <= 1e-12:
        return x.astype(np.float32, copy=False)
    return (float(clip_norm) / norm * x).astype(np.float32, copy=False)


def make_sensitive_blocks_fn(*, rounds: int, clip_norm: float, decay: float = 1.0):
    def sensitive_blocks(x: np.ndarray, label: int) -> np.ndarray:
        del label  # labels are known; the protected object here is the feature contribution.
        x_clip = clip_l2_vector(x, clip_norm=float(clip_norm))
        return np.stack([float(decay ** t) * x_clip for t in range(int(rounds))], axis=0)
    return sensitive_blocks

ATTACK_OBSERVER_MODE = "farthest"
ATTACK_OBSERVER_NODES = observer_nodes(ATTACK_OBSERVER_MODE, attacked_node=ATTACKED_NODE, distances=DIST)
S_ATTACK = selector_matrix(ATTACK_OBSERVER_NODES, num_nodes=NUM_NODES)
H_ATTACK = gossip_transfer_matrix(WS, observer_selector=S_ATTACK, attacked_node=ATTACKED_NODE)
K_ATTACK = gossip_observer_noise_covariance(
    WS,
    observer_selector=S_ATTACK,
    state_noise_stds=PROCESS_NOISE_STD,
    observation_noise_std=0.0,
    jitter=1e-8,
)

mean_fn = make_linear_gaussian_mean_fn(
    transfer_matrix=H_ATTACK,
    sensitive_blocks_fn=make_sensitive_blocks_fn(rounds=ROUNDS, clip_norm=CLIP_NORM),
    base_offset=None,
)

true_record = trial_true_record(trial)
true_mean = np.asarray(mean_fn(true_record.features, int(true_record.label)), dtype=np.float64)

rng = np.random.default_rng(SEED + 1000)
L = np.linalg.cholesky(K_ATTACK)
noise_matrix = L @ rng.normal(size=(K_ATTACK.shape[0], FEATURE_DIM))
observed_view = true_mean + noise_matrix.reshape(-1)

attack = run_linear_gaussian_finite_prior_attack(
    observed_view=observed_view,
    candidate_features=trial.support.X,
    candidate_labels=trial.support.y,
    mean_fn=mean_fn,
    covariance=K_ATTACK,
    prior_weights=trial.support.weights,
    true_record=true_record,
    eta_grid=(0.25, 0.50, 1.00),
    covariance_mode="kron_eye",
)
attack = enrich_attack_result_with_trial(attack, trial)

attack_summary = pd.DataFrame([{
    "observer_mode": ATTACK_OBSERVER_MODE,
    "observer_nodes": str(ATTACK_OBSERVER_NODES),
    "support_size": trial.support.m,
    "baseline_kappa": trial.support.oblivious_kappa,
    "target_source_id": attack.diagnostics.get("target_source_id"),
    "predicted_source_id": attack.diagnostics.get("predicted_source_id"),
    "target_support_position": attack.diagnostics.get("target_support_position"),
    "predicted_prior_index": attack.diagnostics.get("predicted_prior_index"),
    "source_exact_id": attack.metrics.get("source_exact_identification_success"),
    "feature_exact_id": attack.metrics.get("exact_identification_success"),
    "prior_rank": attack.metrics.get("prior_rank"),
    "mse": attack.metrics.get("mse"),
}])
display(attack_summary)


In [ ]:
log_scores = np.asarray(attack.diagnostics["candidate_log_scores"], dtype=float)
shifted = log_scores - float(np.max(log_scores))
posterior = np.exp(shifted)
posterior = posterior / float(np.sum(posterior))
order = np.argsort(-posterior)

score_df = pd.DataFrame({
    "support_position": np.arange(trial.support.m),
    "source_id": list(trial.support.source_ids),
    "posterior_probability": posterior,
    "is_target": np.arange(trial.support.m) == trial.target_support_position,
}).iloc[order]
display(score_df)

fig, ax = plt.subplots(figsize=(7.5, 4.0))
colors = [BALL_COLOR if i == trial.target_support_position else ATTACK_COLOR for i in range(trial.support.m)]
ax.bar(np.arange(trial.support.m), posterior, color=colors)
ax.axhline(1.0 / trial.support.m, color=BASELINE_COLOR, linestyle="--", label="uniform prior")
ax.set_xlabel("support position")
ax.set_ylabel("posterior probability")
ax.set_title("Finite-prior posterior over decentralized observer candidates")
ax.legend()
plt.show()


## 11. Repeated finite-prior attack trials

A single attack is useful for debugging. For reporting, repeat the same canonical setup over multiple anchors/supports/noise draws and compare empirical exact-ID success to the direct Gaussian bound.


In [ ]:
def one_decentralized_attack_trial(
    *,
    trial_id: int,
    bank_seed: int,
    noise_seed: int,
    observer_mode: str,
) -> dict[str, Any]:
    banks_i = find_feasible_replacement_banks(
        X_train=X_train,
        y_train=y_train,
        candidate_sources=[public_source],
        radius=RADIUS,
        min_support_size=PRIOR_SIZE,
        num_banks=1,
        seed=int(bank_seed),
        anchor_selection="large_bank",
        strict=True,
    )
    support_i = select_support_from_bank(
        banks_i[0],
        m=PRIOR_SIZE,
        selection="farthest",
        seed=int(bank_seed),
        draw_index=trial_id,
    )
    pos_i = target_positions_for_support(
        support_i,
        policy="sample",
        num_targets=1,
        seed=int(bank_seed + 777),
    )[0]
    trial_i = make_replacement_trial(
        X_train=X_train,
        y_train=y_train,
        support=support_i,
        target_support_position=pos_i,
    )

    obs = observer_nodes(observer_mode, attacked_node=ATTACKED_NODE, distances=DIST)
    S_A = selector_matrix(obs, num_nodes=NUM_NODES)
    H = gossip_transfer_matrix(WS, observer_selector=S_A, attacked_node=ATTACKED_NODE)
    K = gossip_observer_noise_covariance(
        WS,
        observer_selector=S_A,
        state_noise_stds=PROCESS_NOISE_STD,
        observation_noise_std=0.0,
        jitter=1e-8,
    )
    mean_fn_i = make_linear_gaussian_mean_fn(
        transfer_matrix=H,
        sensitive_blocks_fn=make_sensitive_blocks_fn(rounds=ROUNDS, clip_norm=CLIP_NORM),
    )

    truth = trial_true_record(trial_i)
    mean = np.asarray(mean_fn_i(truth.features, int(truth.label)), dtype=np.float64)
    rng = np.random.default_rng(int(noise_seed))
    L = np.linalg.cholesky(K)
    observed = mean + (L @ rng.normal(size=(K.shape[0], FEATURE_DIM))).reshape(-1)

    attack_i = run_linear_gaussian_finite_prior_attack(
        observed_view=observed,
        candidate_features=trial_i.support.X,
        candidate_labels=trial_i.support.y,
        mean_fn=mean_fn_i,
        covariance=K,
        prior_weights=trial_i.support.weights,
        true_record=truth,
        eta_grid=(0.25, 0.50, 1.00),
        covariance_mode="kron_eye",
    )
    attack_i = enrich_attack_result_with_trial(attack_i, trial_i)

    acct_ball = account_linear_gaussian_observer(
        transfer_matrix=H,
        covariance=K,
        block_sensitivities=block_sensitivities("ball", radius=RADIUS, rounds=ROUNDS),
        parameter_dim=FEATURE_DIM,
        orders=ORDERS,
        radius=RADIUS,
        dp_delta=DP_DELTA,
        attacked_node=ATTACKED_NODE,
        observer=obs,
        covariance_mode="kron_eye",
        method="auto",
    )

    return {
        "trial_id": int(trial_id),
        "observer_mode": observer_mode,
        "observer_nodes": str(obs),
        "support_hash": trial_i.support.support_hash,
        "target_source_id": trial_i.target_source_id,
        "predicted_source_id": attack_i.diagnostics.get("predicted_source_id"),
        "source_exact_id": float(attack_i.metrics.get("source_exact_identification_success", np.nan)),
        "prior_rank": float(attack_i.metrics.get("prior_rank", np.nan)),
        "baseline_kappa": float(trial_i.support.oblivious_kappa),
        "direct_ball_bound": direct_bound_from_accountant(acct_ball, kappa=trial_i.support.oblivious_kappa),
        "transferred_sensitivity": math.sqrt(max(0.0, float(acct_ball.sensitivity_sq))),
    }

attack_rows = []
for t in range(20):
    attack_rows.append(
        one_decentralized_attack_trial(
            trial_id=t,
            bank_seed=SEED + 100 * t,
            noise_seed=SEED + 10_000 + t,
            observer_mode="farthest",
        )
    )

attack_df = pd.DataFrame(attack_rows)
attack_report = pd.DataFrame([{
    "n_trials": len(attack_df),
    "mean_exact_id": float(attack_df.source_exact_id.mean()),
    "baseline_kappa": float(attack_df.baseline_kappa.mean()),
    "mean_direct_ball_bound": float(attack_df.direct_ball_bound.mean()),
    "mean_prior_rank": float(attack_df.prior_rank.mean()),
    "mean_transferred_sensitivity": float(attack_df.transferred_sensitivity.mean()),
}])
display(attack_report.round(4))
display(attack_df.head())


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
vals = [
    float(attack_df.baseline_kappa.mean()),
    float(attack_df.source_exact_id.mean()),
    float(attack_df.direct_ball_bound.mean()),
]
labels = ["prior baseline", "empirical attack", "direct bound"]
ax.bar(labels, vals, color=[BASELINE_COLOR, ATTACK_COLOR, BALL_COLOR])
ax.set_ylim(0.0, 1.02)
ax.set_ylabel("exact-ID probability")
ax.set_title("Decentralized finite-prior exact-ID: empirical vs bound")
for i, v in enumerate(vals):
    ax.text(i, v + 0.025, f"{v:.3f}", ha="center")
plt.show()


## 12. Radius and noise ablations

The Ball bound grows with $r$ until the clipping cap $2C$ saturates. Increasing process noise decreases the observer's whitened signal.


In [ ]:
radius_grid = np.asarray([0.5 * RADIUS, RADIUS, 1.5 * RADIUS, 2.0 * RADIUS], dtype=float)
noise_grid = np.asarray([2.5, 5.0, 10.0, 20.0], dtype=float)
abl_rows = []
obs = observer_nodes("farthest", attacked_node=ATTACKED_NODE, distances=DIST)
S_A = selector_matrix(obs, num_nodes=NUM_NODES)
H = gossip_transfer_matrix(WS, observer_selector=S_A, attacked_node=ATTACKED_NODE)

for r in radius_grid:
    K = gossip_observer_noise_covariance(
        WS,
        observer_selector=S_A,
        state_noise_stds=PROCESS_NOISE_STD,
        jitter=1e-8,
    )
    for mechanism in ["ball", "standard"]:
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=K,
            block_sensitivities=block_sensitivities(mechanism, radius=float(r), rounds=ROUNDS),
            parameter_dim=FEATURE_DIM,
            orders=ORDERS,
            radius=float(r),
            dp_delta=DP_DELTA,
            attacked_node=ATTACKED_NODE,
            observer=obs,
            covariance_mode="kron_eye",
            method="auto",
        )
        abl_rows.append({
            "x_kind": "radius",
            "x_value": float(r),
            "mechanism": mechanism,
            "bound": direct_bound_from_accountant(acct, kappa=support.oblivious_kappa),
            "sensitivity_c": math.sqrt(max(0.0, float(acct.sensitivity_sq))),
        })

for noise in noise_grid:
    K = gossip_observer_noise_covariance(
        WS,
        observer_selector=S_A,
        state_noise_stds=float(noise),
        jitter=1e-8,
    )
    for mechanism in ["ball", "standard"]:
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=K,
            block_sensitivities=block_sensitivities(mechanism, radius=RADIUS, rounds=ROUNDS),
            parameter_dim=FEATURE_DIM,
            orders=ORDERS,
            radius=RADIUS,
            dp_delta=DP_DELTA,
            attacked_node=ATTACKED_NODE,
            observer=obs,
            covariance_mode="kron_eye",
            method="auto",
        )
        abl_rows.append({
            "x_kind": "noise",
            "x_value": float(noise),
            "mechanism": mechanism,
            "bound": direct_bound_from_accountant(acct, kappa=support.oblivious_kappa),
            "sensitivity_c": math.sqrt(max(0.0, float(acct.sensitivity_sq))),
        })

abl = pd.DataFrame(abl_rows)

fig, ax = plt.subplots(figsize=(6.8, 4.0))
for mechanism, color in [("ball", BALL_COLOR), ("standard", STD_COLOR)]:
    sub = abl[(abl.x_kind == "radius") & (abl.mechanism == mechanism)]
    ax.plot(sub.x_value, sub.bound, marker="o", label=mechanism, color=color)
ax.axhline(support.oblivious_kappa, color=BASELINE_COLOR, linestyle="--", label="baseline")
ax.set_xlabel("radius r")
ax.set_ylabel("direct exact-ID bound")
ax.set_ylim(0.0, 1.02)
ax.set_title("Radius ablation")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6.8, 4.0))
for mechanism, color in [("ball", BALL_COLOR), ("standard", STD_COLOR)]:
    sub = abl[(abl.x_kind == "noise") & (abl.mechanism == mechanism)]
    ax.plot(sub.x_value, sub.bound, marker="o", label=mechanism, color=color)
ax.axhline(support.oblivious_kappa, color=BASELINE_COLOR, linestyle="--", label="baseline")
ax.set_xscale("log")
ax.set_xlabel("process noise std")
ax.set_ylabel("direct exact-ID bound")
ax.set_ylim(0.0, 1.02)
ax.set_title("Noise ablation")
ax.legend()
plt.show()


## 13. Decentralized prototype utility/privacy tradeoff

This final section demonstrates a simple decentralized learner: each node computes clipped class-sum prototypes, adds Gaussian noise, and runs deterministic gossip. Counts are treated as public; the protected contribution is the clipped feature vector conditional on label.

This utility experiment is separate from the observer-specific finite-prior attack, but it reports the quantities users usually care about:

$$
\text{accuracy}, \qquad \text{noise scale}, \qquad \text{privacy target}, \qquad \text{consensus disagreement}.
$$


In [ ]:
def gaussian_mechanism_dp_epsilon(*, sensitivity: float, noise_std: float, delta: float, orders: Sequence[float]) -> float:
    eps_rdp = tuple(0.5 * float(a) * (float(sensitivity) / float(noise_std)) ** 2 for a in orders)
    curve = RdpCurve(
        orders=tuple(float(a) for a in orders),
        epsilons=eps_rdp,
        source="single_gaussian_mechanism_rdp",
        radius=None,
    )
    return float(rdp_to_dp(curve, delta=float(delta), source="rdp_to_dp").epsilon)


def calibrate_gaussian_noise(*, sensitivity: float, target_epsilon: float, delta: float, orders: Sequence[float]) -> float:
    lo, hi = 1e-6, max(1.0, float(sensitivity))
    while gaussian_mechanism_dp_epsilon(sensitivity=sensitivity, noise_std=hi, delta=delta, orders=orders) > float(target_epsilon):
        hi *= 2.0
        if hi > 1e6:
            raise RuntimeError("failed to bracket Gaussian noise calibration")
    for _ in range(80):
        mid = math.sqrt(lo * hi)
        eps = gaussian_mechanism_dp_epsilon(sensitivity=sensitivity, noise_std=mid, delta=delta, orders=orders)
        if eps > float(target_epsilon):
            lo = mid
        else:
            hi = mid
    return float(hi)


def prototype_result_row(result: Any) -> dict[str, float]:
    return {
        "accuracy_mean": float(result.accuracy_mean),
        "accuracy_min": float(result.accuracy_min),
        "consensus_disagreement": float(result.consensus_disagreement),
    }

UTILITY_ROUNDS = 10
EPS_GRID = [1.0, 2.0, 4.0, 8.0]
shards = partition_indices_iid(len(X_train), NUM_NODES, seed=SEED)

utility_rows = []
baseline = run_noisy_prototype_gossip(
    X_train=X_train,
    y_train=y_train,
    X_test=X_public,
    y_test=y_public,
    W=W,
    rounds=UTILITY_ROUNDS,
    num_classes=NUM_CLASSES,
    clip_norm=CLIP_NORM,
    noise_std=0.0,
    seed=SEED,
    shards=shards,
)
utility_rows.append({
    "mechanism": "noiseless",
    "epsilon": np.inf,
    "noise_std": 0.0,
    "sensitivity": 0.0,
    **prototype_result_row(baseline),
})

for mechanism in ["ball", "standard"]:
    sensitivity = min(FEATURE_LIPSCHITZ * RADIUS, 2.0 * CLIP_NORM) if mechanism == "ball" else 2.0 * CLIP_NORM
    for eps in EPS_GRID:
        sigma = calibrate_gaussian_noise(
            sensitivity=sensitivity,
            target_epsilon=float(eps),
            delta=DP_DELTA,
            orders=ORDERS,
        )
        result = run_noisy_prototype_gossip(
            X_train=X_train,
            y_train=y_train,
            X_test=X_public,
            y_test=y_public,
            W=W,
            rounds=UTILITY_ROUNDS,
            num_classes=NUM_CLASSES,
            clip_norm=CLIP_NORM,
            noise_std=sigma,
            seed=SEED,
            shards=shards,
        )
        utility_rows.append({
            "mechanism": mechanism,
            "epsilon": float(eps),
            "noise_std": float(sigma),
            "sensitivity": float(sensitivity),
            **prototype_result_row(result),
        })

utility_df = pd.DataFrame(utility_rows)
display(utility_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.0))
for mechanism, color in [("ball", BALL_COLOR), ("standard", STD_COLOR)]:
    sub = utility_df[utility_df.mechanism == mechanism]
    ax.plot(sub.epsilon, sub.accuracy_mean, marker="o", label=mechanism, color=color)
base_acc = float(utility_df.loc[utility_df.mechanism == "noiseless", "accuracy_mean"].iloc[0])
ax.axhline(base_acc, linestyle="--", color=BASELINE_COLOR, label="noiseless baseline")
ax.set_xlabel("target epsilon")
ax.set_ylabel("nearest-prototype accuracy")
ax.set_ylim(0.0, 1.02)
ax.set_title("Decentralized prototype utility/privacy tradeoff")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.0))
for mechanism, color in [("ball", BALL_COLOR), ("standard", STD_COLOR)]:
    sub = utility_df[utility_df.mechanism == mechanism]
    ax.plot(sub.epsilon, sub.noise_std, marker="o", label=mechanism, color=color)
ax.set_xlabel("target epsilon")
ax.set_ylabel("calibrated noise std")
ax.set_yscale("log")
ax.set_title("Noise needed for the same target epsilon")
ax.legend()
plt.show()


## 14. Quantities to report

For a compact decentralized experiment report, include:

1. **Setup:** graph, attacked node, observer set, $r$, $m$, noise, rounds, feature dimension.
2. **Accounting:** local public-transcript $\epsilon$, observer-specific $c_{A \leftarrow j}$, DP $\epsilon$, and direct exact-ID bound.
3. **Attack:** empirical exact-ID success, prior rank, baseline $1/m$, target/predicted source IDs.
4. **Utility:** accuracy, calibrated noise, consensus disagreement.

The table below collects the main quantities from this demo.


In [ ]:
qoi = pd.DataFrame([
    {
        "quantity": "support size m",
        "value": trial.support.m,
    },
    {
        "quantity": "prior baseline kappa",
        "value": trial.support.oblivious_kappa,
    },
    {
        "quantity": "radius r",
        "value": RADIUS,
    },
    {
        "quantity": "observer mode for attack",
        "value": ATTACK_OBSERVER_MODE,
    },
    {
        "quantity": "single-attack source exact-ID",
        "value": attack.metrics.get("source_exact_identification_success"),
    },
    {
        "quantity": "mean repeated exact-ID",
        "value": float(attack_df.source_exact_id.mean()),
    },
    {
        "quantity": "mean direct Ball bound",
        "value": float(attack_df.direct_ball_bound.mean()),
    },
    {
        "quantity": "noiseless prototype accuracy",
        "value": base_acc,
    },
])
display(qoi)


## 15. What is centralized and what is decentralized-specific?

The decentralized project is not totally separate from the finite-prior attack setup. It should reuse the same setup helpers:

```text
CandidateSource
find_feasible_replacement_banks
select_support_from_bank
make_replacement_trial
enrich_attack_result_with_trial
```

Those helpers define $u$, $D^-$, $S$, the hidden target, prior weights, and source-ID bookkeeping.

The decentralized-specific pieces remain separate:

```text
gossip_transfer_matrix
gossip_observer_noise_covariance
account_linear_gaussian_observer
make_linear_gaussian_mean_fn
run_linear_gaussian_finite_prior_attack
run_noisy_prototype_gossip
```

So the right architecture is a shared finite-prior setup layer plus separate observation-model scorers for convex, nonconvex transcript, and decentralized Gaussian observer views.
